In [ ]:
import math
from queue import PriorityQueue

## Cell 2 — Read the graph from input.txt

In [ ]:
coords  = {}   # node id is the key, value = (x, y)
adjlist = {}   # node id is the key, value = list of (neighbor, cost)

with open('input.txt', 'r') as f:
    V = int(f.readline())                      # number of nodes
    for i in range(V):
        strs = f.readline().split()
        nid, x, y = strs[0], int(strs[1]), int(strs[2])
        coords[nid]  = (x, y)                 # store coordinates
        adjlist[nid] = []                      # empty neighbour list

    E = int(f.readline())                      # number of edges
    for i in range(E):
        strs = f.readline().split()
        n1, n2, c = strs[0], strs[1], int(strs[2])
        adjlist[n1].append((n2, c))            # directed edge n1 -> n2

    startnid = f.readline().rstrip()           # start node
    goalnid  = f.readline().rstrip()           # goal node

print('--- Graph ---')
for nid in adjlist:
    print(nid, coords[nid], '--->', adjlist[nid])
print('Start:', startnid, '  Goal:', goalnid)

--- Graph ---
S (6, 0) ---> [('A', 1), ('C', 2), ('D', 4)]
A (6, 0) ---> [('B', 2)]
B (1, 0) ---> [('A', 2), ('G', 1)]
C (2, 0) ---> [('S', 2), ('G', 4)]
D (1, 0) ---> [('G', 4)]
G (0, 0) ---> []
Start: S   Goal: G


## Cell 3 — Heuristic function (Euclidean distance to the goal)

In [ ]:
# ---------------------------------------------------------------
# heuristic(nid, goalnid)
#   Calculates straight-line (Euclidean) distance from node nid
#   to the goal node.  This is an ADMISSIBLE heuristic because
#   straight-line distance never over-estimates the real cost.
# ---------------------------------------------------------------
def heuristic(nid, goalnid):
    x1 = coords[nid][0]       # x coordinate of current node
    y1 = coords[nid][1]       # y coordinate of current node
    x2 = coords[goalnid][0]   # x coordinate of goal node
    y2 = coords[goalnid][1]   # y coordinate of goal node

    # Euclidean distance formula: sqrt( (x2-x1)^2 + (y2-y1)^2 )
    distance = math.sqrt((x2 - x1)**2 + (y2 - y1)**2)
    return distance

# Quick test
print('h(S) =', heuristic('S', 'G'))   # expected: 6.0
print('h(B) =', heuristic('B', 'G'))   # expected: 1.0
print('h(G) =', heuristic('G', 'G'))   # expected: 0.0

h(S) = 6.0
h(B) = 1.0
h(G) = 0.0


## Cell 4 — State class

In [ ]:
# ---------------------------------------------------------------
# State class
#   Each State object represents ONE node we are visiting during
#   the A* search.  It stores:
#     nid    - which node this is (e.g. 'S', 'A', 'G')
#     parent - the State we came from (so we can trace the path)
#     g      - cost from START to this node so far
#     f      - total estimated cost  f = g + h
# ---------------------------------------------------------------
class State:

    # __init__ runs automatically when we create a new State object
    def __init__(self, nid, parent, g, f):
        self.nid    = nid       # node id  (e.g. 'S')
        self.parent = parent    # parent State (None if start node)
        self.g      = g         # cost from start to here
        self.f      = f         # f = g + h  (used for priority)

    # __lt__ tells PriorityQueue how to compare two States.
    # The state with the SMALLER f value has higher priority.
    def __lt__(self, other):
        return self.f < other.f

    # __eq__ checks if two States represent the SAME node.
    def __eq__(self, other):
        return self.nid == other.nid

    # __str__ lets us print a State nicely with print(state)
    def __str__(self):
        parent_id = 'None'
        if self.parent is not None:
            parent_id = self.parent.nid
        return ('Node=' + self.nid +
                '  g=' + str(self.g) +
                '  f=' + str(round(self.f, 2)) +
                '  parent=' + parent_id)

## Cell 5 — A* Search algorithm

In [ ]:
# ---------------------------------------------------------------
# A* Search
#   We use a min-priority queue (minQ) ordered by f = g + h.
#   At each step we pop the State with the lowest f value,
#   and expand it by looking at all its neighbours.
# ---------------------------------------------------------------

# --- Step 1: Create the priority queue and insert the start state ---
minQ = PriorityQueue()

h_start  = heuristic(startnid, goalnid)          # h value for start node
start_state = State(startnid, None, 0, 0 + h_start)   # g=0 at start
minQ.put(start_state)

goal_state = None    # will store the goal State once found

print('=== A* Search begins ===')

# --- Step 2: Main loop — keep going until queue is empty ---
while not minQ.empty():

    # Pop the state with the smallest f value
    curr = minQ.get()
    print('Expanding:', curr)

    # --- Step 3: Check if we reached the goal ---
    if curr.nid == goalnid:
        goal_state = curr
        print('\n*** Goal reached! ***')
        break

    # --- Step 4: Expand neighbours ---
    for tup in adjlist[curr.nid]:
        neighbour_id  = tup[0]                          # neighbour node id
        edge_cost     = tup[1]                          # cost of this edge

        g_new = curr.g + edge_cost                     # new g value
        h_new = heuristic(neighbour_id, goalnid)       # heuristic estimate
        f_new = g_new + h_new                          # f = g + h

        new_state = State(neighbour_id, curr, g_new, f_new)
        minQ.put(new_state)
        print('  Added to queue:', new_state)

# --- Step 5: Trace back the path from goal to start ---
if goal_state is None:
    print('No path found!')
else:
    path = []             # will store node ids in reverse order
    node = goal_state

    while node is not None:
        path.append(node.nid)
        node = node.parent     # go one step back

    path.reverse()             # reverse so it goes start -> goal

    print('\n=== RESULT ===')
    print('Path:', ' -> '.join(path))
    print('Total cost:', goal_state.g)

=== A* Search begins ===
Expanding: Node=S  g=0  f=6.0  parent=None
  Added to queue: Node=A  g=1  f=7.0  parent=S
  Added to queue: Node=C  g=2  f=4.0  parent=S
  Added to queue: Node=D  g=4  f=5.0  parent=S
Expanding: Node=C  g=2  f=4.0  parent=S
  Added to queue: Node=S  g=4  f=10.0  parent=C
  Added to queue: Node=G  g=6  f=6.0  parent=C
Expanding: Node=D  g=4  f=5.0  parent=S
  Added to queue: Node=G  g=8  f=8.0  parent=D
Expanding: Node=G  g=6  f=6.0  parent=C

*** Goal reached! ***

=== RESULT ===
Path: S -> C -> G
Total cost: 6
